### Coleta de dados de Temperatura

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Temperatura   -> variável 2m_temperature, retorna a temperatura em Kelvin, será necessário uma conversão (subtrair -273,15)
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil




In [ ]:
import cdsapi
import sys, os
import xarray as xr
import dask.dataframe as dd
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
import spark_utils as utils 
spark = utils.get_spark_session("Temperatura")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)
print(DATA_PATH_ROOT)

In [ ]:
def get_cdsapi_authentication():
    url = os.getenv("ECMWF_DATASTORES_URL")
    key = os.getenv("ECMWF_DATASTORES_KEY")
    return url, key

def get_t2m(year):

    dataset = "reanalysis-era5-single-levels"
    request = {
        "product_type": ["reanalysis"],
        "variable": ["2m_temperature"],
        "year": [f"{year}"],
        "month": ["01", "02", "03","04", "05", "06","07", "08", "09","10", "11", "12"],
        "day": ["01", "02", "03","04", "05", "06","07", "08", "09","10", "11", "12","13", "14", "15",
                "16", "17", "18","19", "20", "21","22", "23", "24","25", "26", "27","28", "29", "30","31"
        ],
        "time": ["03:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [6, -74, -34, -31]
    }

    # Informações de autenticação estão em:
    # C:\Users\DRT90628\.ecmwfdatastoresrc
    # *** Criar um novo contrato de autenticação deverá ser criado usando um usuário de serviços do Einstein
    url, key = get_cdsapi_authentication()

    client = \
        cdsapi.Client(url = url
                     ,key = key
        )

    ret_download = client.retrieve(dataset, request).download()

    os.rename(ret_download, r"{DATA_PATH_ROOT}\ERA5-temperaturas\arquivos_NC\ERA5_t2m_{year}.nc".format(DATA_PATH_ROOT=DATA_PATH_ROOT, year=year))
    
    return ret_download

def convert_t2m_dataset_to_spark_dataframe(project_path, nc_file_name ):

    with xr.open_dataset(f"{project_path}\{nc_file_name}"
                        ,engine="netcdf4"
                        ) as ds:

        # Transforma o Dataset em um Spark Dataframe
        df_dask        = ds.to_dask_dataframe()
        df_dask_c      = df_dask.compute()
        df_temperatura = spark.createDataFrame(df_dask_c)

    return df_temperatura
    

def transform_data(df_temperatura):
    drop_cols = ["valid_time", "t2m", "number"]

    df_temperatura_final = \
        (df_temperatura
            .withColumns({"data_medicao"    : F.col("valid_time").cast("date")
                         ,"indicador"       : F.lit("temperatura") 
                         ,"valor"           : (F.col("t2m") - F.lit(273.15)).cast("double") # Converte a temperatura de Kelvin para Celsius
                         ,"unidade_medida"  : F.lit("celsius")})
            .drop(*drop_cols)
    )

    df_temperatura_final = \
        (df_temperatura_final
            .select("data_medicao"
                   ,"latitude"
                   ,"longitude"
                   ,"indicador"
                   ,"valor"
                   ,"unidade_medida"))

    return df_temperatura_final


# def write_data_csv(df_temperatura_final, write_path, file_name):
    
#     df_temperatura_final.toPandas().to_csv(f"{write_path}\{file_name}", index=False)

def remove_aux_file(file_name):
    os.remove(file_name)


In [ ]:

with xr.open_dataset(r"C:\Marco Conti\Projetos\Dados\ERA5-temperaturas\arquivos_NC\ERA5_t2m_1990.nc"
                    ,engine="netcdf4"
                    ,chunks={"time": 365
                            ,"latitude": 100
                            ,"longitude": 100 }
                    ) as ds:

#     print(ds['valid_time'])
    
    # Transforma o Dataset em um Spark Dataframe
    df_dask     = ds.to_dask_dataframe()
    df_dask_c   = df_dask.compute()
    df_spark    = spark.createDataFrame(df_dask_c)


In [ ]:
from datetime import datetime 

# years_process = [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]

years_process = range(1991,2027)

for year in years_process:
    start = datetime(2026, 7, 29).now()
    # print("Start download - year : ",year, " - ", start, end="" )
    print("Start transform - year : ",year, " - ", start, end="" )
    
    # Faz download do arquivo de temperaturas do portal Copernicus
    # retorno em formato .nc -> NetCDF (Network Common Data Form)
    ret_download         = get_t2m(year)

    # Converte os dados de temperatura para um Spark Dataframe
    nc_file_name         = f"ERA5_t2m_{year}.nc"
    nc_path              = \
        r"{DATA_PATH_ROOT}\ERA5-temperaturas\arquivos_NC".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
    # df_temperatura       = utils.convert_nc_to_spark_dataframe(spark, nc_path, nc_file_name)
    df_temperatura       = convert_t2m_dataset_to_spark_dataframe(nc_path, nc_file_name)

    # Converte a temperatura de Kelsin para Celsius e adiciona coluna de unidade de medida
    df_temperatura_final = transform_data(df_temperatura)

    # Escreve os dados em formato csv
    csv_file_name = f"ERA5_t2m_{year}.csv"
    csv_path = \
        r"{DATA_PATH_ROOT}\ERA5-temperaturas\arquivos_CSV".format(DATA_PATH_ROOT = DATA_PATH_ROOT)

    # print("\n", f"{csv_path}\{csv_file_name}")

    utils.write_data_csv(df_temperatura_final, csv_path, csv_file_name)

    # remove_aux_file(ret_download)

    finish = datetime(2026, 7, 29).now()

    # print(f" - Download completed: {ret_download} - {finish} - {(finish - start)} \n")
    print(f" - Data transform completed: {csv_file_name} - {finish} - {(finish - start)} \n")
        


In [ ]:
# df_temperatura_final.printSchema()
df_temperatura_final.show(10,False)

Converte os dados baixados do ERA5, que estão em formato NetCDF, para um dataset (xarray.core.dataset.Dataset)

Renomeia nome de colunas e converte o valor da Temperatura recebida do ERA5 está em Kelvin, para converter para Celsius, subtrair 273.15

In [ ]:
df_temperatura.filter("latitude = -34.0 and longitude = -67.0").orderBy("t2m").show(10,False)

In [ ]:
drop_cols = ["valid_time", "t2m", "number"]

df_temperatura_final = \
    (df_temperatura
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("temperatura") 
                     ,"valor": (F.col("t2m") - F.lit(273.15)).cast("double")
                     ,"unidade_medida": F.lit("celsius")})
         .drop(*drop_cols)
    )

df_temperatura_final = \
    (df_temperatura_final
        .select("data_medicao"
               ,"latitude"
               ,"longitude"
               ,"indicador"
               ,"valor"
               ,"unidade_medida"))

df_temperatura_final.printSchema()

df_temperatura_final.show()

In [ ]:
# df_temperatura_final.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_t2m_temperatura.csv", index=False)

df_temperatura_final.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\mais_einstein\\dados\\ERA5-temperaturas\\2025\\ERA5_temperatura.parquet")
